In [4]:
import pyart
import numpy as np

In [5]:
f = "/Volumes/Rain/original_radar_precip_single/20230306/212519.mdv"

In [6]:
radar = pyart.io.read_grid_mdv(
    f,
    file_field_names=True,
)

print(radar.fields["rate"].get("units", "Units not specified"))

Units not specified


In [7]:
print("File:", f)
print("\nRATE FIELD METADATA:")
for key, value in radar.fields["rate"].items():
    if key != "data":
        print(f"{key}: {value}")

data = radar.fields["rate"]["data"]

print("\nDATA STATISTICS:")
print("Min:", data.min())
print("Max:", data.max())
print("Mean:", data.mean())
print("Percentiles:", np.ma.percentile(data.compressed(), [50, 90, 95, 99]) if hasattr(np.ma, "percentile") else np.percentile(data.compressed(), [50, 90, 95, 99]))

File: /Volumes/Rain/original_radar_precip_single/20230306/212519.mdv

RATE FIELD METADATA:
_FillValue: -9999.0

DATA STATISTICS:
Min: 0.0
Max: 155.87291
Mean: 0.7860679789182982
Percentiles: [0.         2.30515177 3.8896749  9.16775607]


In [8]:
print("GLOBAL METADATA:")
print(radar.metadata)

print("\nALL FIELDS:")
print(radar.fields.keys())

print("\nRATE FIELD:")
print({
    k: v
    for k, v in radar.fields["rate"].items()
    if k != "data"
})

GLOBAL METADATA:
{'Conventions': 'CF/Radial instrument_parameters', 'version': '1.3', 'title': '', 'institution': '', 'references': '', 'source': 'Precip accumulation using ZR and reflectivity composite.\n  ZR coeff, expon: 300, 1.4\n  Hail threshold: 60\n  Low dBZ threshold: 20\n  min_ht, max_ht: 1.5, 30\n', 'history': '', 'comment': '', 'instrument_name': 'Computed by applying ZR function to dBZ data and integrating.'}

ALL FIELDS:
dict_keys(['rate', 'precip'])

RATE FIELD:
{'_FillValue': -9999.0}


In [9]:
rate = radar.fields["rate"]["data"][0]
precip = radar.fields["precip"]["data"][0]

valid = (
    (~np.ma.getmaskarray(rate))
    & (~np.ma.getmaskarray(precip))
    & (rate > 0)
    & (precip > 0)
)

ratio = np.ma.array(
    precip / np.ma.maximum(rate, 1e-10),
    mask=~valid,
)

print("Rate max:", rate.max())
print("Precip max:", precip.max())

print("Median precip/rate ratio:", np.ma.median(ratio))
print("Expected ratio for 5 min:", 5 / 60)

Rate max: 155.87291
Precip max: 12.383237
Median precip/rate ratio: 0.07944444454536063
Expected ratio for 5 min: 0.08333333333333333


In [11]:
print("Radar time metadata:")
print(radar.time)

print("\nRadar metadata:")
print(radar.metadata)

# print("\nFile name:")
# print(f.name)

print("\nDecoded radar time:")
print(radar)

Radar time metadata:
{'units': 'seconds since 2023-03-06T21:20:33Z', 'standard_name': 'time', 'long_name': 'Time of grid', 'calendar': 'gregorian', 'data': array([286])}

Radar metadata:
{'Conventions': 'CF/Radial instrument_parameters', 'version': '1.3', 'title': '', 'institution': '', 'references': '', 'source': 'Precip accumulation using ZR and reflectivity composite.\n  ZR coeff, expon: 300, 1.4\n  Hail threshold: 60\n  Low dBZ threshold: 20\n  min_ht, max_ht: 1.5, 30\n', 'history': '', 'comment': '', 'instrument_name': 'Computed by applying ZR function to dBZ data and integrating.'}

Decoded radar time:


In [15]:
from pathlib import Path
import pyart
from netCDF4 import num2date

# Select an MDV file
f = sorted(Path(
    "/Users/matthewwright/Downloads/single_precip_sample"
).glob("*/*.mdv"))[1]

radar = pyart.io.read_grid_mdv(
    f,
    file_field_names=True,
)

# Decode timestamp
radar_time = num2date(
    radar.time["data"][0],
    units=radar.time["units"],
    calendar=radar.time.get("calendar", "standard"),
    only_use_cftime_datetimes=False,
)

print("Filename:", f.name)
print("Directory date:", f.parent.name)
print("Decoded radar timestamp:", radar_time)

print("\nRaw time metadata:")
print(radar.time)

print("\nRadar metadata:")
print(radar.metadata)

print("\nRadar start/end time attributes:")
print("time_start:", getattr(radar, "time_start", None))
print("time_end:", getattr(radar, "time_end", None))

Filename: 035531.mdv
Directory date: 20211016
Decoded radar timestamp: 2021-10-16 03:55:31

Raw time metadata:
{'units': 'seconds since 2021-10-16T03:50:11Z', 'standard_name': 'time', 'long_name': 'Time of grid', 'calendar': 'gregorian', 'data': array([320])}

Radar metadata:
{'Conventions': 'CF/Radial instrument_parameters', 'version': '1.3', 'title': '', 'institution': '', 'references': '', 'source': 'Precip accumulation using ZR and reflectivity composite.\n  ZR coeff, expon: 300, 1.4\n  Hail threshold: 60\n  Low dBZ threshold: 20\n  min_ht, max_ht: 1.5, 30\n', 'history': '', 'comment': '', 'instrument_name': 'Computed by applying ZR function to dBZ data and integrating.'}

Radar start/end time attributes:
time_start: None
time_end: None


In [13]:
import pandas as pd
radar_time = pd.Timestamp(
    radar.time["units"].split("since", 1)[1].strip()
).tz_localize(None)

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyart
from netCDF4 import num2date

# ------------------------------------------------------------
# Path to 24-hour accumulation MDV files
# ------------------------------------------------------------

base_dir = Path(
    "/Volumes/Rain/original_radar_precip_24h"
)

# Find a representative MDV file
files = sorted(base_dir.rglob("*.mdv"))
files = [f for f in files if str(f.name)[0] is not "."]

print(f"Found {len(files)} MDV files")

if not files:
    raise FileNotFoundError(f"No MDV files found in {base_dir}")

f = files[100]

print("\n" + "=" * 65)
print("FILE INFORMATION")
print("=" * 65)

print("File:", f)
print("Filename:", f.name)
print("Parent directory:", f.parent.name)

# ------------------------------------------------------------
# Read radar
# ------------------------------------------------------------

radar = pyart.io.read_grid_mdv(
    f,
    file_field_names=True,
)

# ------------------------------------------------------------
# Time information
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TIME INFORMATION")
print("=" * 65)

print("Raw time metadata:")
print(radar.time)

radar_time = num2date(
    radar.time["data"][0],
    units=radar.time["units"],
    calendar=radar.time.get("calendar", "standard"),
    only_use_cftime_datetimes=False,
)

epoch = pd.Timestamp(
    radar.time["units"].split("since", 1)[1].strip()
)

print("\nEpoch:", epoch)
print("Decoded timestamp:", radar_time)
# print("Time difference:", pd.Timestamp(radar_time) - epoch)

print("\nRadar time_start:", getattr(radar, "time_start", None))
print("Radar time_end:", getattr(radar, "time_end", None))

# ------------------------------------------------------------
# Global metadata
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("GLOBAL METADATA")
print("=" * 65)

for key, value in radar.metadata.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# Available fields
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("AVAILABLE FIELDS")
print("=" * 65)

print(list(radar.fields.keys()))

for field_name, field in radar.fields.items():

    print(f"\n--- FIELD: {field_name} ---")

    for key, value in field.items():
        if key != "data":
            print(f"{key}: {value}")

    data = np.ma.asarray(field["data"])

    valid = data.compressed()

    print("Shape:", data.shape)
    print("Valid pixels:", valid.size)
    print("Masked pixels:", np.ma.count_masked(data))

    if valid.size > 0:

        print("Minimum:", np.min(valid))
        print("Maximum:", np.max(valid))
        print("Mean:", np.mean(valid))

        print(
            "Percentiles [50, 90, 95, 99]:",
            np.percentile(valid, [50, 90, 95, 99])
        )

        print("Nonzero pixels:", np.sum(valid > 0))

# ------------------------------------------------------------
# Grid information
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("GRID INFORMATION")
print("=" * 65)

# print("Grid shape (nz, ny, nx):", radar.shape)

print("X range:", radar.x["data"].min(), radar.x["data"].max())
print("Y range:", radar.y["data"].min(), radar.y["data"].max())

print("X spacing:", np.diff(radar.x["data"])[:5])
print("Y spacing:", np.diff(radar.y["data"])[:5])

print("\nProjection:")
print(radar.projection)

print("\nRadar origin:")
print("Longitude:", radar.origin_longitude["data"])
print("Latitude:", radar.origin_latitude["data"])

Found 11656 MDV files

FILE INFORMATION
File: /Volumes/Rain/original_radar_precip_24h/20201120/000006.mdv
Filename: 000006.mdv
Parent directory: 20201120

TIME INFORMATION
Raw time metadata:
{'units': 'seconds since 2020-11-20T00:00:00Z', 'standard_name': 'time', 'long_name': 'Time of grid', 'calendar': 'gregorian', 'data': array([6])}

Epoch: 2020-11-20 00:00:00+00:00
Decoded timestamp: 2020-11-20 00:00:06

Radar time_start: None
Radar time_end: None

GLOBAL METADATA
Conventions: CF/Radial instrument_parameters
version: 1.3
title: 
institution: 
references: 
source: Precip accumulation using ZR and reflectivity composite.
  ZR coeff, expon: 200, 1.455
  Hail threshold: 55
  Low dBZ threshold: 10
  min_ht, max_ht: 0, 30

history: 
comment: 
instrument_name: Computed by summing single-file precip values.

AVAILABLE FIELDS
['rate', 'precip']

--- FIELD: rate ---
_FillValue: -9999.0
Shape: (1, 800, 800)
Valid pixels: 4713
Masked pixels: 635287
Minimum: 0.0
Maximum: 0.0
Mean: 0.0
Percentil